# Titanic V1: O Baseline (Random Forest)
Este foi o arquivo experimental base do nosso projeto, idealmente formatado para atingir a nossa primeira métrica sólida no Kaggle. 

Aqui utilizamos ferramentas estatísticas diretas: imputação usando Médias e Medianas condicionais, e uma árvore de classificação pura.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. Carregando os Dados
# Utilizaremos a pasta local onde devem residir os datasets originais do Kaggle
df_train = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

### Análise Visual Primária

In [ ]:
# Visualizando Correlações Iniciais de Sobrevivência
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.countplot(data=df_train, x='Sex', hue='Survived', palette='pastel')
plt.title('Sobreviventes por Sexo')

plt.subplot(1, 2, 2)
sns.countplot(data=df_train, x='Pclass', hue='Survived', palette='pastel')
plt.title('Sobreviventes por Classe')
plt.tight_layout()
plt.show()

### EDA e Limpeza Inicial (Feature Engineering Simples)

In [ ]:
# Extração do Título e Imputação de Idades Faltantes (Age)
df_train['Title'] = df_train['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
df_test['Title'] = df_test['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

df_train['Title'] = df_train['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df_train['Title'] = df_train['Title'].replace('Mlle', 'Miss').replace('Ms', 'Miss').replace('Mme', 'Mrs')

df_test['Title'] = df_test['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df_test['Title'] = df_test['Title'].replace('Mlle', 'Miss').replace('Ms', 'Miss').replace('Mme', 'Mrs')

# Preenchendo buracos de Idade com a regra do Título Escalar
df_train['Age'] = df_train['Age'].fillna(df_train.groupby('Title')['Age'].transform('median'))
df_test['Age'] = df_test['Age'].fillna(df_test.groupby('Title')['Age'].transform('median'))

# Preenchendo variáveis fáceis faltantes na base
df_train['Embarked'] = df_train['Embarked'].fillna(df_train['Embarked'].mode()[0])
df_test['Fare'] = df_test['Fare'].fillna(df_test['Fare'].median())

# Tamanho da Família Agrupado Simples
df_train['FamilySize'] = df_train['SibSp'] + df_train['Parch'] + 1
df_test['FamilySize'] = df_test['SibSp'] + df_test['Parch'] + 1

### Preparando para a Inteligência Artificial (Encoding)

In [ ]:
# Jogando dados ruidosos fora (No V1 descartamos o Ticket e a Cabin)
cols_to_drop = ['PassengerId', 'Cabin', 'Ticket', 'Name', 'SibSp', 'Parch']
df_train = df_train.drop(cols_to_drop, axis=1)
passenger_id_test = df_test['PassengerId']
df_test = df_test.drop(cols_to_drop, axis=1)

# Transformando Variáveis em Expressões Numéricas (One-Hot-Encoding)
df_train['Sex'] = df_train['Sex'].map({'male': 0, 'female': 1})
df_test['Sex'] = df_test['Sex'].map({'male': 0, 'female': 1})

df_train = pd.get_dummies(df_train, columns=['Embarked', 'Title'], drop_first=True)
df_test = pd.get_dummies(df_test, columns=['Embarked', 'Title'], drop_first=True)

# Importante: Garantir que o Teste tem as mesmas colunas exatas que o Treino
missing_cols = set(df_train.columns) - set(df_test.columns)
for c in missing_cols:
    if c != 'Survived': # O Teste obviamente não tem acesso ao Survived
        df_test[c] = 0
        
# Reescrevendo a ordem as colunas se necessário
df_test = df_test[[c for c in df_train.columns if c != 'Survived']]

### Modelagem e Acurácia

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Separando Máquinas
X = df_train.drop('Survived', axis=1)
y = df_train['Survived']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

# Métrica da V1 (Aproximadamente 82.12% local)
y_pred = rf_model.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print(f"Nossa Nota Final da Versão 1 com RF: {acc * 100:.2f}%")

# Treinamento Força-Total e Exportação 
rf_model.fit(X, y)
test_predictions = rf_model.predict(df_test)

submission = pd.DataFrame({'PassengerId': passenger_id_test, 'Survived': test_predictions})
submission.to_csv('../submission.csv', index=False)
print("E o arquivo final foi sobrescrito/gerado: submission.csv")